# Storms beyond the fold

Power-flow benchmarks keep only cases that have an AC solution, so a case without one is never scored. For contingency
screening that case matters most: it may be a grid past voltage collapse, or a case the solver cannot settle
(`fold_check` follows every storm by continuation to tell the two apart). This notebook builds 120 clustered N-20 outages
("storms"), some solved at the actual load and some not, and looks at what GENCO says about them.

- For every storm, the largest load factor at which one of three Newton starts converges (the nose), to ±0.25 %, from a
  march-and-bisect search; `fold_check` locates the fold by continuation and compares the two.
- GENCO from 15 % below to 15 % beyond each nose, and at the actual load.
- At every point: the AC solution where one exists, GENCO alone, DC power flow, GENCO followed by a Newton check, and
  Newton from flat and DC starts.

Runtime: T4 GPU, about 0.3 GB of downloads, 25 to 40 minutes. Output: `storms.zip`.

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # the revision every run used (unchanged since 1 October 2026)
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # the commit every run used (branch head from 1 to 14 October 2026)
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "storms"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVEL_BASE, BASE_PARTS = 2, [0, 1]   # ordinary N-2 operating points (two data blocks) as storm base points
N_BASE = 120                  # base points; one storm each
STORM_TOTAL = 20              # lines out after the storm (the 2 already out + 18 nearest to a random centre)
LAM_STEP = 0.02               # marching step of the load factor while looking for the nose
LAM_WIDTH = 0.0025            # bisection width: the nose is located to +/- this
LAM_MIN, LAM_CAP = 0.3, 2.0   # give up below / above these load factors
GRID_STEP, GRID_HALF = 0.01, 15   # GENCO is run at nose + GRID_STEP * j for j = -GRID_HALF .. GRID_HALF, and at the actual load
N_SHOWCASE = 6                # storms whose full voltage profiles are saved
BATCH = 8
TOL, TOL_SCREEN, MAX_NR = 1e-8, 1e-2, 6   # Newton tolerance; the screening check's tolerance and maximum steps
MAX_DOWNLOAD_GB = 2
SEED = 0

In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""J1 core: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the step-1/6 core."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""K1 core: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC truth along lambda up to the point where the AC equations stop having a solution (the
"nose"), a signature that the nose is genuine, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the step-1/6 core and power_flow_jacobian from the J1 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor at which a Newton start converges, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # no start converges at lambda = 1: go down until one does
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Drops toward 0 at a genuine nose."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from gridfm_graphkit.datasets.globals import PD_H, QD_H, PG_H

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# N-2 scenarios that are hard for GENCO: not used as storm base points, so the base points are ordinary
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def pick_storm(d, extra, rng):
    """`extra` more lines to remove, the nearest (in hops) to a random centre bus, keeping the grid in one piece.
    Returns branch-row indices, the storm radius in hops and the centre bus; (None, None, centre) if impossible."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None, center
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)

## 1. Download: model and two N-2 data blocks (about 0.3 GB)

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
for p, s in all_files:
    m = RAW3_PAT.search(p) if p.startswith(f"data/test_{LEVEL_BASE}/") else None
    if m and int(m.group(2)) in BASE_PARTS:
        plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

## 2. Base points: ordinary N-2 operating points with their stored AC solutions

In [ ]:
def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}


base_disk = scenario_ids(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"))
base_pool = np.array([s for s in base_disk if int(s) not in HARD2], dtype=np.int64)
BASE_IDS = np.sort(RNG.choice(base_pool, min(N_BASE, len(base_pool)), replace=False)).astype(np.int64)
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
NB = next(iter(BASE.values()))["nb"]
tot = np.array([d["Pd"].sum() for d in BASE.values()])
print(f"{len(BASE)} N-{LEVEL_BASE} operating points as storm base points | {NB} buses | "
      f"total load {tot.min() / 1000:.1f}-{tot.max() / 1000:.1f} GW")

## 3. GENCO: graphs, model, and a check against the stored solutions

In [ ]:
# [GPU] build graphs of the base points (used for the model check)

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

BUS_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"), BASE_IDS)
GEN_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"), BASE_IDS)
BR_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data"), BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")


def build_graph(sid, base, Pd=None, Qd=None, removed_lines=()):
    """GENCO's input graph for base point `base` with extra lines removed and (optionally) other loads."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    if Pd is not None:
        b["Pd"], b["Qd"] = np.asarray(Pd, float), np.asarray(Qd, float)
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    return TRANSFORM(g)


def scaled_graph(g, lam):
    """The same graph with every load and every non-slack generator scaled by lam (the slack's generation is a
    masked input, 0, and stays 0; loads and generation are divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    return h


BASE_GRAPHS = [build_graph(int(s), int(s)) for s in BASE_IDS]
print(f"{len(BASE_GRAPHS)} base graphs built")

In [ ]:
# [GPU] load model

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)

In [ ]:
# [GPU] run GENCO on a list of graphs; check the model against the stored AC solutions of the base points

def run_genco(graphs, bs=BATCH):
    """GENCO's (Vm, Va in radians) for each graph, in order."""
    out = [None] * len(graphs)
    for i, h in enumerate(graphs):
        h["scenario_id"] = torch.tensor([i], dtype=torch.long)
    for b in DataLoader(graphs, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad():
            o = forward(MODEL, b)["bus"].float()
        ptr = b["bus"].ptr.cpu().tolist()
        for g, i in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            out[int(i)] = (o[ptr[g]:ptr[g + 1], VM_OUT].cpu().numpy().astype(np.float64),
                           o[ptr[g]:ptr[g + 1], VA_OUT].cpu().numpy().astype(np.float64))
    return out


t0 = time.perf_counter()
BASE_OUT = run_genco(BASE_GRAPHS)
T_FWD_MS = 1000 * (time.perf_counter() - t0) / len(BASE_GRAPHS)
errs = np.array([np.max(np.abs(vm - BASE[int(s)]["Vm"])[BASE[int(s)]["PQ"]]) for s, (vm, va) in zip(BASE_IDS, BASE_OUT)])
print(f"GENCO on {len(BASE_IDS)} base points: worst-bus voltage error median {np.median(errs):.4f} p.u., "
      f"95th percentile {np.percentile(errs, 95):.4f} p.u. (expected about 0.001 and 0.004) | {T_FWD_MS:.0f} ms per graph incl. loading")
if np.median(errs) > 0.02 and NB > 100:
    raise RuntimeError("GENCO's error on ordinary N-2 cases is far above the known level: the model did not load correctly; check the installation and the downloaded checkpoint.")

## 4. Storms: 18 more lines out around a random centre, grid kept in one piece

Each storm is solved from three starts (the pre-storm solution, flat, DC) with up to 30 Newton iterations.
Storms that no start solves are kept: they are the point of this notebook.

In [ ]:
STORM_RNG = np.random.default_rng(SEED + 1)
CASES, not_built = [], 0
sid = 910_000_000
for j, b_id in enumerate(BASE_IDS):
    d = BASE[int(b_id)]
    extra = STORM_TOTAL - int((~d["on"]).sum())
    rem, radius, centre = pick_storm(d, extra, STORM_RNG) if extra > 0 else (None, None, -1)
    if rem is None:
        not_built += 1; continue
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][rem] = False
    d2["Va_dc"] = dc_angles(d2)
    case = LambdaCase(d2)
    V1, who = case.solve_any(1.0, [("base", case.state(d["Vm"], d["Va"])), ("flat", case.flat), ("dc", case.dc)], 30, TOL)
    if V1 is not None:
        d2["Vm"], d2["Va"] = np.abs(V1), np.degrees(np.angle(V1))
    sid += 1
    CASES.append({"sid": sid, "base": int(b_id), "T": STORM_TOTAL, "centre": centre, "radius_hops": radius,
                  "removed_lines": d["line"][rem].astype(int).tolist(), "d": d2, "case": case,
                  "solvable_at_1": V1 is not None, "V1": V1, "solved_from": who})
n_unsolv = sum(not c["solvable_at_1"] for c in CASES)
print(f"storms built: {len(CASES)} (could not build {not_built}) | solved at the actual load: {len(CASES) - n_unsolv} | "
      f"with NO AC solution from any start (base, flat, DC; 30 Newton iterations): {n_unsolv}")

## 5. The nose of every storm

All loads and all non-slack generation are scaled by a load factor λ (constant power factor, set-points unchanged,
slack takes the rest). λ_max is the largest factor at which a Newton start converges: march up in steps of 2 % from a solvable
point, bisect the first failure to ±0.25 %; at every failure the flat and DC starts are tried as well. The AC truth is
then computed on a grid below the nose. A genuine nose shows as the Jacobian's smallest singular value dropping toward
zero. A storm that no start solves at the actual load has λ_max < 1 (a load *deficit*); one with a solution has λ_max > 1
(a *margin*).

In [ ]:
t0 = time.perf_counter()
for c in CASES:
    case = c["case"]
    res = find_lambda_max(case, c["V1"], step=LAM_STEP, width=LAM_WIDTH, lam_min=LAM_MIN, lam_cap=LAM_CAP, tol=TOL)
    c.update(lam_max=res["lam_max"], lam_first=res["lam_first"], capped=res["capped"], n_solves=res["n_solves"],
             path_starts=[w for _, w in res["path"]])
    c["has_nose"] = bool(np.isfinite(res["lam_max"]) and not res["capped"])
    c["truth"] = {}
    if c["solvable_at_1"]:
        c["truth"][1.0] = c["V1"]
    c["monotone"] = np.nan; c["sigma_nose"] = c["sigma_far"] = np.nan
    if c["has_nose"]:
        grid_below = [round(res["lam_max"] - GRID_STEP * j, 8) for j in range(1, GRID_HALF + 1)]
        T = truth_below_nose(case, res["lam_max"], res["V_max"], grid_below, tol=TOL)
        c["truth"].update(T); c["truth"][res["lam_max"]] = res["V_max"]
        c["monotone"] = bool(len(T) == len(grid_below))
        try:
            c["sigma_nose"] = sigma_min(case.jacobian(res["V_max"]))
            far = grid_below[min(9, len(grid_below) - 1)]
            c["sigma_far"] = sigma_min(case.jacobian(T[far])) if far in T else np.nan
        except Exception as exc:
            print("sigma_min failed for case", c["sid"], repr(exc)[:120])
    lams = ([round(res["lam_max"] + GRID_STEP * j, 8) for j in range(-GRID_HALF, GRID_HALF + 1)] if c["has_nose"] else [])
    if not any(abs(l - 1.0) < 1e-9 for l in lams):
        lams.append(1.0)
    c["lams"] = sorted(l for l in lams if l >= LAM_MIN)
summ = pd.DataFrame([{"sid": c["sid"], "solvable_at_1": c["solvable_at_1"], "has_nose": c["has_nose"], "lam_max": c["lam_max"],
                      "monotone": c["monotone"], "sigma_nose": c["sigma_nose"], "sigma_far": c["sigma_far"],
                      "sigma_ratio": c["sigma_nose"] / c["sigma_far"], "n_solves": c["n_solves"]} for c in CASES])
print(f"nose search done in {time.perf_counter() - t0:.0f} s | cases with a nose below {LAM_CAP}: {int(summ.has_nose.sum())} of {len(summ)}")
with_nose = summ[summ.has_nose]
if len(with_nose):
    print(f"lambda_max: storms WITHOUT a solution at the actual load: median {with_nose[~with_nose.solvable_at_1].lam_max.median():.3f} "
          f"(deficit {100 * (1 - with_nose[~with_nose.solvable_at_1].lam_max.median()):.1f} %); "
          f"storms WITH a solution: median {with_nose[with_nose.solvable_at_1].lam_max.median():.3f} "
          f"(margin {100 * (with_nose[with_nose.solvable_at_1].lam_max.median() - 1):.1f} %)")
    print(f"continuation monotone (every grid point below the nose solved): {100 * with_nose.monotone.mean():.0f} % | "
          f"smallest singular value of the Jacobian at the nose / 0.1 below it: median {with_nose.sigma_ratio.median():.2f} "
          f"(a genuine nose: well below 1)")
    display(summ.drop(columns=["sid"]).describe().T[["count", "mean", "50%", "min", "max"]])

## 6. GENCO through and beyond the nose

In [ ]:
# [GPU] GENCO at every load factor of every storm, through and beyond the nose

t0 = time.perf_counter()
VMG, VAG = {}, {}
for c in CASES:
    g0 = build_graph(c["sid"], c["base"], c["d"]["Pd"], c["d"]["Qd"], c["removed_lines"])
    out = run_genco([scaled_graph(g0, lam) for lam in c["lams"]])
    VMG[c["sid"]] = {lam: vm for lam, (vm, va) in zip(c["lams"], out)}
    VAG[c["sid"]] = {lam: va for lam, (vm, va) in zip(c["lams"], out)}
n_fwd = sum(len(c["lams"]) for c in CASES)
print(f"GENCO run at {n_fwd} (storm, load factor) points in {time.perf_counter() - t0:.0f} s")

## 7. What everyone says at every point

Per (storm, load factor): the AC truth where it exists; GENCO alone (voltage limits, overloads with the 4.54 %
conformal margin of notebook `overload_margin`); DC alone (overloads with its 13.7 % margin; DC has no voltages and a solution for
every case); GENCO + the screening check (at most 6 Newton steps, stop below 0.01 p.u.); GENCO + a full
Newton solve (30 iterations); and the classical solve (full Newton from a flat or DC start).

In [ ]:
# What the AC truth, GENCO, DC and the checks say at every (storm, load factor) point
LINE_MARGIN, DC_LINE_MARGIN = 0.0454, 0.137     # scenario-level conformal margins from notebook `overload_margin` (GENCO) and the DC rule
rows = []
t0 = time.perf_counter()
for c in CASES:
    case, d = c["case"], c["d"]
    lam_max = c["lam_max"] if c["has_nose"] else np.nan
    weak_true = int(case.decisions(lam_max, c["truth"][lam_max])["weak_bus"]) if c["has_nose"] else -1
    c["weak_true"] = weak_true
    for lam in c["lams"]:
        vm, va = VMG[c["sid"]][lam], VAG[c["sid"]][lam]
        Vg = case.state(vm, np.degrees(va))
        g = case.decisions(lam, Vg)
        gm = case.decisions(lam, Vg, margin_line=LINE_MARGIN)
        r = {"sid": c["sid"], "solvable_at_1": c["solvable_at_1"], "lam": lam, "lam_max": lam_max,
             "lam_rel": lam - lam_max if c["has_nose"] else np.nan, "beyond_nose": bool(c["has_nose"] and lam > lam_max + 1e-9),
             "at_actual_load": abs(lam - 1.0) < 1e-9, "truth_exists": lam in c["truth"]}
        r.update({f"genco_{k}": v for k, v in g.items()})
        r["genco_n_over_margin"] = gm["n_over"]
        r["genco_flags"] = bool(g["n_low_v"] > 0 or g["n_high_v"] > 0 or gm["n_over"] > 0)
        r["genco_vm_weak_true"] = float(np.abs(Vg[weak_true])) if weak_true >= 0 else np.nan
        ld_dc = case.dc_loading(lam)
        r["dc_n_over"] = int((ld_dc >= 1.0).sum()); r["dc_n_over_margin"] = int((ld_dc >= 1.0 - DC_LINE_MARGIN).sum())
        r["dc_flags"] = bool(r["dc_n_over_margin"] > 0)
        # the checks: full Newton (KLU, 30 iterations) from GENCO's answer, from flat and from DC; the
        # screening check (at most MAX_NR Python Newton steps, stop at TOL_SCREEN) from GENCO's answer
        Vk, okk, itk, sk = case.solve(lam, Vg, 30, TOL)
        r.update({"klu_from_genco_converged": bool(okk), "klu_from_genco_iters": itk, "klu_from_genco_ms": 1000 * sk})
        r["klu_from_flat_converged"] = bool(case.solve(lam, case.flat, 30, TOL)[1])
        r["klu_from_dc_converged"] = bool(case.solve(lam, case.dc, 30, TOL)[1])
        tr = newton_trace(case.Y, case.S(lam), Vg, case.pv, case.pq, tol=TOL, max_it=MAX_NR)
        j = next((j for j in range(1, len(tr)) if tr[j][1] <= TOL_SCREEN), None)
        r["check_steps"] = j if j is not None else MAX_NR
        r["check_passed"] = j is not None
        r["check_mismatch_end_pu"] = tr[-1][1]
        r["check_ms"] = 1000 * sum(t for _, _, t in tr)
        Vc = tr[j][0] if j is not None else tr[-1][0]
        chk = case.decisions(lam, Vc)
        r["check_flags"] = bool(chk["n_low_v"] > 0 or chk["n_high_v"] > 0 or chk["n_over"] > 0)
        r["check_min_vm"] = chk["min_vm"]
        if lam in c["truth"]:
            Vt = c["truth"][lam]
            t = case.decisions(lam, Vt)
            r.update({f"true_{k}": v for k, v in t.items() if k in ("min_vm", "weak_bus", "n_low_v", "n_high_v", "n_over", "max_loading")})
            r["true_flags"] = bool(t["n_low_v"] > 0 or t["n_high_v"] > 0 or t["n_over"] > 0)
            err = np.abs(np.abs(Vg) - np.abs(Vt))
            r["err_vm_max"] = float(err[d["PQ"]].max())
            r["err_vm_weak_true"] = float(err[weak_true]) if weak_true >= 0 else np.nan
            r["true_vm_weak_true"] = float(np.abs(Vt[weak_true])) if weak_true >= 0 else np.nan
            r["weak_same"] = bool(g["weak_bus"] == t["weak_bus"])
            r["weak_hops"] = hop_distance(d, g["weak_bus"], t["weak_bus"])
            over_t = case.loading(Vt) >= 1.0
            r["missed_overloads"] = int((over_t & (case.loading(Vg) < 1.0 - LINE_MARGIN)).sum())
        rows.append(r)
SWEEP = pd.DataFrame(rows)
# storms with no solution from the base, flat and DC starts but solved by Newton from GENCO's answer: not collapsing
# after all (GENCO rescued them); their nose from the continuation is then not trustworthy either
RESCUED = set(SWEEP[SWEEP.at_actual_load & ~SWEEP.solvable_at_1 & SWEEP.klu_from_genco_converged].sid)
for c in CASES:
    c["rescued"] = c["sid"] in RESCUED
SWEEP["rescued"] = SWEEP.sid.isin(RESCUED)
SWEEP.loc[SWEEP.rescued, ["lam_rel", "lam_max"]] = np.nan
SWEEP.loc[SWEEP.rescued, "beyond_nose"] = False
SWEEP.to_csv(Path(OUT_DIR) / "sweep.csv", index=False)
print(f"{len(SWEEP)} points analysed in {time.perf_counter() - t0:.0f} s | storms rescued by Newton from GENCO's answer: {len(RESCUED)}")

# per-storm table
CASES_T = pd.DataFrame([{"sid": c["sid"], "base": c["base"], "T": c["T"], "centre_bus": c["centre"], "radius_hops": c["radius_hops"],
                         "removed_lines": json.dumps(c["removed_lines"]), "solvable_at_1": c["solvable_at_1"], "solved_from": c["solved_from"],
                         "has_nose": c["has_nose"], "lam_max": c["lam_max"], "lam_first": c["lam_first"], "capped": c["capped"],
                         "monotone": c["monotone"], "sigma_nose": c["sigma_nose"], "sigma_far": c["sigma_far"],
                         "n_solves": c["n_solves"], "weak_bus_true": c["weak_true"], "rescued_by_genco_start": c["rescued"]} for c in CASES])
CASES_T.to_csv(Path(OUT_DIR) / "cases.csv", index=False)

In [ ]:
# 1. At the actual load, for the storms with NO AC solution: what would each screen have told the operator?
U = SWEEP[SWEEP.at_actual_load & ~SWEEP.solvable_at_1 & ~SWEEP.rescued]
S1 = SWEEP[SWEEP.at_actual_load & SWEEP.solvable_at_1]
pct = lambda m: float(100 * np.mean(m)) if len(m) else np.nan
if len(U):
    SCREEN = pd.DataFrame([
        {"screen": "GENCO alone (voltage limits; overloads with the 4.54 % conformal margin)",
         "'calm grid' (%)": pct(~U.genco_flags), "flags a violation (%)": pct(U.genco_flags), "alarm: no solution (%)": 0.0,
         "median lowest voltage shown (p.u.)": U.genco_min_vm.median()},
        {"screen": "DC power flow alone (overloads with its 13.7 % margin; it has no voltages and always a solution)",
         "'calm grid' (%)": pct(~U.dc_flags), "flags a violation (%)": pct(U.dc_flags), "alarm: no solution (%)": 0.0,
         "median lowest voltage shown (p.u.)": np.nan},
        {"screen": f"GENCO + screening check (at most {MAX_NR} Newton steps, stop below {TOL_SCREEN} p.u.), alarm if the check does not get there",
         "'calm grid' (%)": pct(U.check_passed & ~U.check_flags), "flags a violation (%)": pct(U.check_passed & U.check_flags),
         "alarm: no solution (%)": pct(~U.check_passed), "median lowest voltage shown (p.u.)": U.check_min_vm.median()},
        {"screen": "GENCO + full Newton (30 iterations), alarm if it does not converge",
         "'calm grid' (%)": 0.0, "flags a violation (%)": 0.0, "alarm: no solution (%)": pct(~U.klu_from_genco_converged),
         "median lowest voltage shown (p.u.)": np.nan},
    ])
    SCREEN.to_csv(Path(OUT_DIR) / "screen_unsolvable.csv", index=False)
    print(f"{len(U)} storms with no AC solution at the actual load (plus {len(RESCUED)} rescued by Newton from GENCO's answer, "
          f"not counted); for comparison {len(S1)} storms with a solution. The classical solve (full Newton from a flat or DC start) reports "
          f"'diverged' on all {len(U)} by construction.")
    display(SCREEN)
    print(f"Cost of the alarm: the screening check runs {U.check_steps.median():.0f} steps ({U.check_ms.median():.1f} ms) on these storms "
          f"against {S1.check_steps.median():.0f} steps ({S1.check_ms.median():.1f} ms) on solvable storms; "
          f"full Newton from GENCO's answer: {U.klu_from_genco_ms.median():.1f} ms vs {S1.klu_from_genco_ms.median():.1f} ms.")
    print(f"GENCO's own picture of these collapsing storms: lowest voltage median {U.genco_min_vm.median():.3f} p.u. "
          f"(5th-95th pct {U.genco_min_vm.quantile(0.05):.3f}-{U.genco_min_vm.quantile(0.95):.3f}); "
          f"overloads predicted: median {U.genco_n_over.median():.0f}; low-voltage buses predicted: median {U.genco_n_low_v.median():.0f}")
    if len(S1):
        print(f"Solvable storms at the actual load for comparison: true lowest voltage median {S1.true_min_vm.median():.3f}, "
              f"GENCO {S1.genco_min_vm.median():.3f}; GENCO flags {100 * S1.genco_flags.mean():.0f} % of them, truth flags {100 * S1.true_flags.mean():.0f} %")
else:
    print("No storm without an AC solution at the actual load: the screen table is empty.")

# 2. Beyond the nose, pooled over all storms with a nose (how the picture changes with the deficit)
B = SWEEP[SWEEP.beyond_nose]
if len(B):
    bins = pd.cut(B.lam_rel, [0, 0.025, 0.05, 0.1, 0.2])
    BEYOND = B.groupby(bins, observed=True).agg(points=("sid", "count"), genco_calm_pct=("genco_flags", lambda x: 100 * (~x).mean()),
                                                dc_calm_pct=("dc_flags", lambda x: 100 * (~x).mean()),
                                                check_passes_pct=("check_passed", lambda x: 100 * x.mean()),
                                                klu_converges_pct=("klu_from_genco_converged", lambda x: 100 * x.mean()),
                                                genco_min_vm=("genco_min_vm", "median"), resP_p99_MW=("genco_resP_p99_MW", "median"))
    BEYOND.to_csv(Path(OUT_DIR) / "beyond_nose.csv")
    print("\nBeyond the nose (all storms with a nose), by how far beyond:"); display(BEYOND)

In [ ]:
# 3. Does GENCO's answer change at the nose, and does its own residual know?
W = SWEEP[SWEEP.lam_max.notna()].copy()
glide = []
for sid_, grp in W.groupby("sid"):
    grp = grp.sort_values("lam")
    before = grp[(grp.lam_rel > -0.05 - 1e-9) & (grp.lam_rel <= 1e-9)]
    after = grp[(grp.lam_rel > 1e-9) & (grp.lam_rel <= 0.05 + 1e-9)]
    if len(before) >= 3 and len(after) >= 3:
        sg_b = np.polyfit(before.lam, before.genco_vm_weak_true, 1)[0]
        sg_a = np.polyfit(after.lam, after.genco_vm_weak_true, 1)[0]
        st_b = np.polyfit(before.lam, before.true_vm_weak_true, 1)[0]
        drop_true = before.true_vm_weak_true.iloc[0] - before.true_vm_weak_true.iloc[-1]
        drop_genco = before.genco_vm_weak_true.iloc[0] - before.genco_vm_weak_true.iloc[-1]
        glide.append({"sid": sid_, "slope_genco_before": sg_b, "slope_genco_after": sg_a, "slope_true_before": st_b,
                      "genco_after_over_before": sg_a / sg_b if sg_b != 0 else np.nan,
                      "genco_drop_over_true_drop_last_0.05": drop_genco / drop_true if drop_true != 0 else np.nan,
                      "err_vm_weak_at_nose": grp[np.abs(grp.lam_rel) < 1e-9].err_vm_weak_true.iloc[0] if (np.abs(grp.lam_rel) < 1e-9).any() else np.nan,
                      "true_vm_weak_at_nose": grp[np.abs(grp.lam_rel) < 1e-9].true_vm_weak_true.iloc[0] if (np.abs(grp.lam_rel) < 1e-9).any() else np.nan,
                      "genco_vm_weak_at_nose": grp[np.abs(grp.lam_rel) < 1e-9].genco_vm_weak_true.iloc[0] if (np.abs(grp.lam_rel) < 1e-9).any() else np.nan})
GLIDE = pd.DataFrame(glide)
GLIDE.to_csv(Path(OUT_DIR) / "glide.csv", index=False)
if len(GLIDE):
    print(f"At the true weakest bus, over the last 5 % of load before the nose, GENCO's voltage drops {100 * GLIDE['genco_drop_over_true_drop_last_0.05'].median():.0f} % "
          f"of the true drop (median; IQR {100 * GLIDE['genco_drop_over_true_drop_last_0.05'].quantile(0.25):.0f}-{100 * GLIDE['genco_drop_over_true_drop_last_0.05'].quantile(0.75):.0f} %).")
    print(f"GENCO's slope after the nose / before it: median {GLIDE.genco_after_over_before.median():.2f} "
          f"(IQR {GLIDE.genco_after_over_before.quantile(0.25):.2f}-{GLIDE.genco_after_over_before.quantile(0.75):.2f}); 1 = no sign of the nose at all.")
    print(f"At the nose: true voltage at the weakest bus median {GLIDE.true_vm_weak_at_nose.median():.3f} p.u., GENCO {GLIDE.genco_vm_weak_at_nose.median():.3f} p.u.; "
          f"GENCO's error there median {GLIDE.err_vm_weak_at_nose.median():.3f} p.u.")

# residual as a signal: points beyond the nose against points at least 5 % below it
pos = W.beyond_nose.to_numpy(); neg = (W.lam_rel <= -0.05 + 1e-9).to_numpy()
sel = pos | neg
SIGNALS = {}
for name, score in [("resP_p99_MW", W.genco_resP_p99_MW), ("resP_max_MW", W.genco_resP_max_MW), ("mismatch_pu", W.genco_mismatch_pu),
                    ("minus_min_vm", -W.genco_min_vm), ("n_low_v", W.genco_n_low_v)]:
    SIGNALS[name] = {"auroc_beyond_vs_margin5": auroc(score.to_numpy()[sel], pos[sel])}
if len(U) and len(S1):
    A1 = SWEEP[SWEEP.at_actual_load]
    for name, score in [("resP_p99_MW", A1.genco_resP_p99_MW), ("resP_max_MW", A1.genco_resP_max_MW), ("mismatch_pu", A1.genco_mismatch_pu),
                        ("minus_min_vm", -A1.genco_min_vm), ("n_low_v", A1.genco_n_low_v)]:
        SIGNALS[name]["auroc_unsolvable_vs_solvable_at_actual_load"] = auroc(score.to_numpy(), (~A1.solvable_at_1).to_numpy())
SIG = pd.DataFrame(SIGNALS).T
SIG.to_csv(Path(OUT_DIR) / "signals.csv")
print("\nCan GENCO's own output tell a case beyond the nose from one with margin? (AUROC; 0.5 = no, 1 = perfectly)"); display(SIG)

# the weakest bus near the nose
N0 = W[(np.abs(W.lam_rel) < 1e-9) & W.truth_exists]
if len(N0):
    print(f"\nAt the nose, GENCO's lowest-voltage bus is the true one in {100 * N0.weak_same.mean():.0f} % of storms; "
          f"within 2 lines in {100 * (N0.weak_hops <= 2).mean():.0f} %; median distance {N0.weak_hops.median():.0f} lines. "
          f"GENCO's worst voltage error at the nose: median {N0.err_vm_max.median():.3f} p.u.; it misses {N0.missed_overloads.sum()} of "
          f"{int(N0.true_n_over.sum())} true overloads there (with the conformal margin).")

In [ ]:
# Full voltage profiles along the load factor for a few storms (for the figures and the write-up)
with_nose = [c for c in CASES if c["has_nose"]]
uns = sorted([c for c in with_nose if not c["solvable_at_1"]], key=lambda c: c["lam_max"])
sol = sorted([c for c in with_nose if c["solvable_at_1"]], key=lambda c: c["lam_max"])
pick = lambda L, n: [L[i] for i in sorted(set(np.linspace(0, len(L) - 1, n).round().astype(int)))] if L else []
SHOW = pick(uns, min(N_SHOWCASE // 2, len(uns))) + pick(sol, min(N_SHOWCASE - min(N_SHOWCASE // 2, len(uns)), len(sol)))
show = {}
for c in SHOW:
    lams = np.array(c["lams"])
    vm_g = np.stack([VMG[c["sid"]][l] for l in c["lams"]]).astype(np.float32)
    vm_t = np.full_like(vm_g, np.nan)
    for i, l in enumerate(c["lams"]):
        if l in c["truth"]:
            vm_t[i] = np.abs(c["truth"][l])
    show[f"{c['sid']}_lams"] = lams; show[f"{c['sid']}_vm_genco"] = vm_g; show[f"{c['sid']}_vm_true"] = vm_t
    show[f"{c['sid']}_meta"] = np.array([c["lam_max"], float(c["solvable_at_1"]), c["weak_true"], c["centre"]])
    show[f"{c['sid']}_removed"] = np.array(c["removed_lines"])
np.savez_compressed(Path(OUT_DIR) / "showcase.npz", sids=np.array([c["sid"] for c in SHOW]), **show)
print("showcase storms:", [(c["sid"], round(c["lam_max"], 3), "unresolved at actual load" if not c["solvable_at_1"] else "solvable") for c in SHOW])

## 8. Figures

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, c in zip(axes.ravel(), SHOW):
    lams = np.array(c["lams"]); wb = c["weak_true"]
    vg = np.array([VMG[c["sid"]][l][wb] for l in c["lams"]])
    vt = np.array([np.abs(c["truth"][l][wb]) if l in c["truth"] else np.nan for l in c["lams"]])
    grid = np.abs(lams - c["lam_max"]) <= GRID_STEP * GRID_HALF + 1e-9     # the grid around the nose, joined by lines
    ax.plot(lams[grid], vt[grid], "o-", color="#7f7f7f", ms=3, label="AC truth (stops at the nose)")
    ax.plot(lams[grid], vg[grid], "s-", color="#1f5fa8", ms=3, label="GENCO")
    ax.plot(lams[~grid], vt[~grid], "o", color="#7f7f7f", ms=4); ax.plot(lams[~grid], vg[~grid], "s", color="#1f5fa8", ms=4)
    ax.axvline(c["lam_max"], color="#c8102e", ls="--", lw=1); ax.axvline(1.0, color="k", ls=":", lw=1)
    ax.set_title(f"storm {c['sid'] - 910_000_000}: {'unresolved at actual load' if not c['solvable_at_1'] else 'solvable at actual load'}\n"
                 f"nose at load factor {c['lam_max']:.3f}; bus {wb}", fontsize=9)
    ax.set_xlabel("load factor (1 = actual load)"); ax.set_ylabel(f"voltage at the true weakest bus (p.u.)"); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8, frameon=False)
for ax in axes.ravel()[len(SHOW):]:
    ax.axis("off")
fig.suptitle("Through and beyond the nose: the AC solution stops, GENCO keeps answering", y=1.0)
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / "fig_pv_curves.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
if len(U):
    labels = ["GENCO alone", "DC alone", f"GENCO + check\n({MAX_NR} steps, {TOL_SCREEN} p.u.)", "GENCO + full\nNewton (30 it.)"]
    vals = [100 * (~U.genco_flags).mean(), 100 * (~U.dc_flags).mean(), 100 * (U.check_passed & ~U.check_flags).mean(), 0.0]
    axes[0].bar(range(4), vals, color=["#1f5fa8", "#e07b20", "#2ca25f", "#2ca25f"])
    axes[0].set_xticks(range(4)); axes[0].set_xticklabels(labels, fontsize=8); axes[0].set_ylim(0, 100)
    axes[0].set_ylabel("% of collapsing storms waved through (no alarm)"); axes[0].set_title(f"{len(U)} storms with no AC solution at the actual load", fontsize=10)
    axes[0].grid(alpha=0.3, axis="y")
if len(W):
    W["bin"] = W.lam_rel.round(3)
    med = W.groupby("bin", observed=True).agg(x=("lam_rel", "median"), r50=("genco_resP_p99_MW", "median"),
                                              r25=("genco_resP_p99_MW", lambda v: v.quantile(0.25)), r75=("genco_resP_p99_MW", lambda v: v.quantile(0.75)),
                                              g=("genco_min_vm", "median"), t=("true_min_vm", "median"))
    axes[1].fill_between(med.x, med.r25, med.r75, color="#1f5fa8", alpha=0.2); axes[1].plot(med.x, med.r50, "o-", color="#1f5fa8", ms=3)
    axes[1].axvline(0, color="#c8102e", ls="--", lw=1); axes[1].set_yscale("log")
    axes[1].set_xlabel("load factor minus the nose"); axes[1].set_ylabel("GENCO's 99th-pct active residual (MW)")
    axes[1].set_title(f"GENCO's own signal (AUROC beyond vs 5 % margin: {SIG.loc['resP_p99_MW', 'auroc_beyond_vs_margin5']:.2f})", fontsize=10); axes[1].grid(alpha=0.3)
    axes[2].plot(med.x, med.t, "o-", color="#7f7f7f", ms=3, label="AC truth"); axes[2].plot(med.x, med.g, "s-", color="#1f5fa8", ms=3, label="GENCO")
    axes[2].axvline(0, color="#c8102e", ls="--", lw=1); axes[2].set_xlabel("load factor minus the nose"); axes[2].set_ylabel("lowest voltage (p.u., median over storms)")
    axes[2].set_title("The grid's weakest voltage, truth and GENCO", fontsize=10); axes[2].legend(frameon=False, fontsize=8); axes[2].grid(alpha=0.3)
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / "fig_screen_and_signal.png", dpi=150); plt.show()

## 9. Summary and output files

In [ ]:
V = {}
if len(summ[summ.has_nose]):
    wn = summ[summ.has_nose]
    V["NOSE_IS_GENUINE"] = bool(wn.monotone.mean() >= 0.9 and (wn.sigma_ratio < 0.5).mean() >= 0.8)
if len(GLIDE):
    V["GENCO_SHOWS_NO_NOSE"] = bool(GLIDE["genco_drop_over_true_drop_last_0.05"].median() < 0.5 and 0.5 <= GLIDE.genco_after_over_before.median() <= 2.0)
if len(U):
    V["GENCO_ALONE_BLIND"] = bool((~U.genco_flags).mean() >= 0.5)
    V["DC_ALONE_BLIND"] = bool((~U.dc_flags).mean() >= 0.5)
    V["CHECK_CATCHES_IT"] = bool((~U.check_passed).mean() >= 0.95)
    V["CHECK_PASSES_BUT_FLAGS"] = bool((U.check_passed & U.check_flags).mean() >= 0.5)
    V["FULL_NEWTON_FROM_GENCO_CATCHES_IT"] = bool((~U.klu_from_genco_converged).mean() >= 0.95)
if len(SIG):
    V["RESIDUAL_KNOWS"] = bool(SIG.loc["resP_p99_MW", "auroc_beyond_vs_margin5"] >= 0.8)
print("VERDICT")
for k, v in V.items():
    print(f"  {k}: {v}")
if V.get("GENCO_SHOWS_NO_NOSE"):
    print("  -> GENCO's voltages glide through the point where no start converges any more: it still returns a state.")
if V.get("GENCO_ALONE_BLIND"):
    print("  -> Used alone, GENCO would wave through most storms that collapse the grid.")
if V.get("CHECK_CATCHES_IT"):
    print("  -> The verify step catches them, provided a 'did not converge' exit raises an alarm.")
elif len(U):
    print(f"  -> The screening check PASSES on {100 * U.check_passed.mean():.0f} % of collapsing storms (a 1 MW tolerance cannot tell "
          f"'just beyond the nose' from 'just before'); of those, the checked state still flags a violation on {100 * (U.check_passed & U.check_flags).mean():.0f} % of all collapsing storms.")
summary = {"config": {"N_BASE": N_BASE, "STORM_TOTAL": STORM_TOTAL, "LAM_STEP": LAM_STEP, "LAM_WIDTH": LAM_WIDTH, "GRID_STEP": GRID_STEP,
                      "GRID_HALF": GRID_HALF, "LAM_CAP": LAM_CAP, "TOL": TOL, "TOL_SCREEN": TOL_SCREEN, "MAX_NR": MAX_NR, "SEED": SEED,
                      "LINE_MARGIN": LINE_MARGIN, "DC_LINE_MARGIN": DC_LINE_MARGIN},
           "n_cases": len(CASES), "n_unsolvable_at_actual_load": int(n_unsolv), "n_with_nose": int(summ.has_nose.sum()),
           "verdict": V, "signals": SIGNALS, "gpu": str(DEVICE), "lightsim2grid": bool(HAVE_KLU), "forward_ms_incl_loading": T_FWD_MS,
           "torch_scatter_replacement": bool(globals().get("USED_SHIM", False)), "platform": platform.platform()}
(Path(OUT_DIR) / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).iterdir()):
        zf.write(p, arcname=f"{Path(OUT_DIR).name}/{p.name}")
print(f"\nSaved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB).")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass